# Terzaghi 2D Consolidation: FEM (FEniCSx)

This notebook runs the 2D Terzaghi consolidation FEM solver and produces the key diagnostic plots using the `src/plotting/terzaghi_2d/plot.py` module.

## Problem definition
- Domain: rectangle `[-W, W] × [-H, 0]` — width 2W, depth H.
- `y = 0` is the ground surface (drained boundary, `u = 0`).
- Initial condition: Boussinesq 2D strip-load pore pressure distribution.
- Time integration: implicit Euler (backward Euler).
- Governing equation: `∂u/∂t = Cv ∇²u`

## Outputs
1. Initial pore pressure contour on the 2D mesh.
2. Pore pressure contour at an intermediate and final time.
3. Surface settlement profile at multiple time steps.
4. Settlement trough evolution over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# set up notebook imports to use project source package under src/
%load_ext autoreload
%autoreload 2
import os
import sys
project_root = os.path.abspath(os.path.join('..'))
sys.path.insert(0, project_root)

from src.geotech_consolidation.models.terzaghi_2d.fem import Get_terzaghi2D_FEA
from src.plotting.terzaghi_2d.plot import (
    Get_Pore_Pressure_Contour,
    Get_Settlement_Surface_Plot,
    Get_Settlement_Surface_History_Plot,
)

# --- Parameters ---
H          = 5          # depth (m)
W          = H          # half-width (m); domain is [-W, W] x [-H, 0]
nx         = 50         # elements per direction
load       = 100        # applied load (kPa)
base       = 2          # loaded width for Boussinesq IC (m)
Cv         = 2e-7       # coefficient of consolidation (m^2/s)
Mv         = 5e-4       # coefficient of volume compressibility (1/kPa)
T          = (60*60*24) * 10000   # final time (s) — ~27 years
time_steps = 50

time = np.linspace(0, T / (60*60*24), num=time_steps)   # time axis in days

# FEM Solution
Run the 2D solver. Returns:
- `settlement_surface` — (time_steps, nX) settlement at each surface x-column
- `u_hist` — (time_steps, n_nodes) pore pressure at every node
- `unique_X` — x-coordinates for the settlement profile
- `node_X`, `node_Y` — mesh node coordinates for contour plots

In [ ]:
settlement_surface, u_hist, unique_X, node_X, node_Y = Get_terzaghi2D_FEA(
    H, W, nx, load, T, time_steps, Cv, Mv, base
)

print(f"u_hist shape:          {u_hist.shape}          (time_steps x n_nodes)")
print(f"settlement_surface:    {settlement_surface.shape}      (time_steps x nX)")
print(f"unique_X range:        [{unique_X.min():.1f}, {unique_X.max():.1f}] m")
print(f"Max pore pressure:     {u_hist.max():.2f} kPa")
print(f"Max settlement:        {settlement_surface.max()*1000:.2f} mm")

# Initial Pore Pressure (t = 0)
Boussinesq 2D strip-load distribution across the domain.

In [ ]:
fig, ax = Get_Pore_Pressure_Contour(u_hist, node_X, node_Y, time_idx=0, time=time)
plt.tight_layout()
plt.show()

# Pore Pressure at Mid-Time and Final Time

In [ ]:
mid_idx  = time_steps // 2
final_idx = time_steps - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

import matplotlib.tri as mtri
tri = mtri.Triangulation(node_X, node_Y)
u_min, u_max = u_hist.min(), u_hist.max()

for ax, idx, label in zip(axes, [mid_idx, final_idx], ["Mid-time", "Final time"]):
    cs = ax.tricontourf(tri, u_hist[idx], levels=20, vmin=u_min, vmax=u_max, cmap="YlOrRd")
    fig.colorbar(cs, ax=ax, label="u (kPa)")
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("Depth (m)")
    ax.set_title(f"{label} — t = {time[idx]:.0f} days")

plt.tight_layout()
plt.show()

# Surface Settlement Profile
Settlement trough across the surface width at the final time step.

In [ ]:
fig, ax = Get_Settlement_Surface_Plot(
    settlement_surface, unique_X, time_idx=final_idx, time=time
)
plt.tight_layout()
plt.show()

# Settlement Trough Evolution Over Time
Multiple time-step curves overlaid — shows how the trough deepens and widens.

In [ ]:
fig, ax = Get_Settlement_Surface_History_Plot(
    settlement_surface, unique_X, time, n_curves=8
)
plt.tight_layout()
plt.show()

# Centre-Line Settlement vs Time
Settlement directly below the load (x ≈ 0) over time — analogous to `settlement_history` in the 1D model.

In [ ]:
# find the column index closest to x = 0
centre_idx = np.argmin(np.abs(unique_X))

fig, ax = plt.subplots()
ax.plot(time, -settlement_surface[:, centre_idx], label=f"x = {unique_X[centre_idx]:.2f} m")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Settlement (m)")
ax.set_title("Centre-Line Surface Settlement vs Time")
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Max centre-line settlement: {settlement_surface[:, centre_idx].max()*1000:.2f} mm")